In [1]:
!git clone https://github.com/YPolina/Medicine.git

Cloning into 'Medicine'...
remote: Enumerating objects: 1431, done.
remote: Counting objects: 100% (83/83), done.
remote: Compressing objects: 100% (43/43), done.
remote: Total 1431 (delta 36), reused 67 (delta 28), pack-reused 1348 (from 3)
Receiving objects: 100% (1431/1431), 474.76 MiB | 3.97 MiB/s, done.
Resolving deltas: 100% (110/110), done.
Updating files: 100% (103/103), done.


In [1]:
%cd ./Medicine/BELKA/training

/content/Medicine/BELKA/training


In [3]:
!pip install -r ../requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 311.4/311.4 MB 4.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.3/34.3 MB 42.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 495.4/495.4 kB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 113.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 88.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 56.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [6]:
import sys
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import pandas as pd

sys.path.append(os.path.abspath(os.path.join('..')))

sys.modules.pop("functionality.models", None)
sys.modules.pop("functionality.data_preparation", None)
from functionality.data_preparation import EmbDataset, train_model
from functionality.models import ChemBertaBinaryClassifierLightning

from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
import pytorch_lightning as pl
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint
from pytorch_lightning.loggers import CSVLogger
from transformers import AutoTokenizer, AutoModel

import torch
import pickle
from tqdm import tqdm
import gc
from torch.cuda.amp import autocast

In [3]:
binds_0 = pd.read_parquet("../intermediates/downsampled_0_50_mln")
binds_1 = pd.read_parquet("../intermediates/1_class")
final_data = pd.concat([binds_0, binds_1], axis=0).sample(frac=1, random_state=42).reset_index(drop=True)
del binds_1
del binds_0

In [4]:
def compute_and_save_embeddings(model_name, data, save_path, batch_size):
    """
    Compute embeddings for a list of SMILES strings in batches and save them

    Args:
        model_name (str): Pretrained model name
        data (pd.Series):data with SMILES strings
        save_path (str): Path to save computed embeddings
    """
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    with open(save_path, "wb") as f:
        for i in tqdm(range(0, len(data), batch_size), desc="Computing Embeddings"):
            batch_smiles = data[i : i + batch_size]
            batch_smiles = batch_smiles.tolist()
            tokens = tokenizer(batch_smiles, padding=True, truncation=True, max_length=512, return_tensors="pt")
            tokens = {k: v.to(device) for k, v in tokens.items()}

            with torch.no_grad(), autocast():
                outputs = model(**tokens)

            batch_embeddings = outputs.last_hidden_state[:, 0, :].cpu()
            pickle.dump(batch_embeddings, f)

            del batch_smiles, tokens, outputs, batch_embeddings
            gc.collect()
            torch.cuda.empty_cache()

    print(f"Embeddings saved to {save_path}")
    return save_path

In [5]:
protein_names = final_data.protein_name.unique()
save_dir= "../intermediates/embeddings"
models = {
    "ChemBert": "seyonec/PubChem10M_SMILES_BPE_450k",
    "MolFormer": "ibm/MoLFormer-XL-both-10pct"
}
batch_size=1000

for model_name, model_ in models.items():
  for protein_name in protein_names:
      print(f"Training model for protein: {protein_name}")

      protein_data = final_data[final_data.protein_name == protein_name]
      train_data, val_data = train_test_split(final_data[final_data.protein_name == protein_name], test_size=0.1, random_state=42)

      train_embeddings_path = os.path.join(save_dir, f"{protein_name}_{model_name}_train_embeddings.pkl")
      val_embeddings_path = os.path.join(save_dir, f"{protein_name}__{model_name}_val_embeddings.pkl")

      train_embeddings_path = compute_and_save_embeddings(model_, train_data['molecule_smiles'], train_embeddings_path, batch_size)
      val_embeddings_path = compute_and_save_embeddings(model_, val_data["molecule_smiles"], val_embeddings_path, batch_size)


Training model for protein: BRD4


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Computing Embeddings:   0%|          | 1/3414 [02:39<151:40:36, 159.99s/it]


KeyboardInterrupt: 

In [ ]:
protein_names = final_data.protein_name.unique()
save_dir= "../intermediates/embeddings"
model_name="ibm/MoLFormer-XL-both-10pct"
batch_size=10000
model_ = MolFormer
for protein_name in protein_names:
    print(f"Training model for protein: {protein_name}")

    protein_data = final_data[final_data.protein_name == protein_name]
    train_data, val_data = train_test_split(protein_data, test_size=0.1, random_state=42)

    train_embeddings_path = os.path.join(save_dir, f"{protein_name}_{model_}_train_embeddings.pkl")
    val_embeddings_path = os.path.join(save_dir, f"{protein_name}__{model_}_val_embeddings.pkl")

    train_embeddings_path = compute_and_save_embeddings(model_name, train_data['molecule_smiles'], train_embeddings_path, batch_size)
    val_embeddings_path = compute_and_save_embeddings(model_name, val_data["molecule_smiles"], val_embeddings_path, batch_size)


In [ ]:
def train_model(final_data, model_name="seyonec/PubChem10M_SMILES_BPE_450k", batch_size=10000, save_dir="../intermediates"):
    protein_names = final_data.protein_name.unique()

    for protein_name in protein_names:
        print(f"Training model for protein: {protein_name}")

        protein_data = final_data[final_data.protein_name == protein_name]
        train_data, val_data = train_test_split(protein_data, test_size=0.1, random_state=42)

        train_embeddings_path = os.path.join(save_dir, f"{protein_name}_train_embeddings.pkl")
        val_embeddings_path = os.path.join(save_dir, f"{protein_name}_val_embeddings.pkl")

        train_embeddings_path = compute_and_save_embeddings(model_name, train_data['molecule_smiles'], train_embeddings_path, batch_size)
        val_embeddings_path = compute_and_save_embeddings(model_name, val_data["molecule_smiles"], val_embeddings_path, batch_size)

        train_dataset = EmbDataset(train_data, train_embeddings_path)
        val_dataset = EmbDataset(val_data, val_embeddings_path)

        train_loader = DataLoader(train_dataset, batch_size=512, shuffle=True, num_workers=4, pin_memory=True)
        val_loader = DataLoader(val_dataset, batch_size=512, shuffle=False, num_workers=4, pin_memory=True)
        logger = CSVLogger("logs", name=model_name)
        early_stopping = EarlyStopping(monitor="val_loss", patience=3, mode="min")
        checkpoint_callback = ModelCheckpoint(
            dirpath="../checkpoints",
            filename=f"{protein_name}_{model_name}-{{epoch}}-{{val_loss:.4f}}",
            monitor="val_loss",
            save_top_k=1,
            mode="min",
            save_last=True,
            verbose=True,
        )

        trainer = pl.Trainer(
            max_epochs=20,
            accelerator="auto",
            devices=1,
            log_every_n_steps=2,
            callbacks=[early_stopping, checkpoint_callback],
            logger=logger,
        )


        chemberta_model = ChemBertaBinaryClassifierLightning()
        trainer.fit(chemberta_model, train_loader, val_loader)

        os.makedirs("../models", exist_ok=True)
        trainer.save_checkpoint(f"../models/{protein_name}_{model_name}.ckpt")

        print(f"Completed training for protein: {protein_name}")

        del train_data, val_data, train_dataset, val_dataset, train_loader, val_loader, chemberta_model
        gc.collect()

In [ ]:
train_model(final_data)

Training model for protein: BRD4


Computing Embeddings:   0%|          | 0/342 [00:00<?, ?it/s]

: 